# Practical 9: Build a Simple Classifier to Detect Attacks
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO5

### Objective:
Train, benchmark, and evaluate supervised classifiers (Logistic Regression, Random Forest, XGBoost) along with an unsupervised Isolation Forest for zero-day attack discovery. Compute precision, recall, F1, ROC-AUC, and PR-AUC, and export production model artifacts.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd
from sklearn.model_selection import train_test_split

from pipeline.parser import parse_file
from pipeline.cleaner import clean
from pipeline.labeler import load_rules, label_requests
from pipeline.features import add_request_features
from models.train import prepare_feature_matrix, train_all_models
from models.evaluate import evaluate_models, predict_event

LOG_PATH = Path('../tests/fixtures/sample_access.log')
RULES_PATH = Path('../rules/web_rules.yaml')


## 2. Prepare Balanced Training & Evaluation Splits

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)
rules = load_rules(RULES_PATH)
labeled_df = label_requests(cleaned_df, rules)
feat_df = add_request_features(labeled_df)

X, y, feature_cols = prepare_feature_matrix(feat_df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Training instances: {len(X_train)} | Test instances: {len(X_test)}')
print('Feature dimensions:', len(feature_cols))


## 3. Train Multi-Architecture Model Suite

In [ ]:
bundle = train_all_models(
    X_train, y_train, feature_cols,
    artifact_dir='../models/artifacts'
)
print(f"Model version {bundle['version']} trained successfully.")
print('Models in bundle:', list(bundle.keys()))


## 4. Benchmark & Comparative Model Evaluation

In [ ]:
metrics_df = evaluate_models(bundle, X_test, y_test, reports_dir='../reports')
print('Comprehensive Evaluation Matrix:')
metrics_df


## 5. Live Inference Demonstration (`predict_event`)

In [ ]:
probs, anomalies = predict_event(feat_df, bundle_or_path=bundle)
feat_df['ml_prob'] = probs
feat_df['anomaly_score'] = anomalies

print('Sample Predictions on Telemetry:')
feat_df[['raw_url', 'label', 'ml_prob', 'anomaly_score']].head(10)


### Conclusion:
Practical 9 produced an explainable hybrid detection engine combining supervised classification (Random Forest / XGBoost) with unsupervised outlier detection (Isolation Forest), achieving high precision and recall on web attacks.